# Avaliação do RAG — CLT

## Objetivo

Avaliar a qualidade das respostas geradas pelo pipeline RAG (Retrieval-Augmented Generation) sobre a CLT brasileira.

## Metodologia

Utilizamos duas abordagens complementares:

1. **Avaliação automática por palavra-chave (Keyword Accuracy):** verifica se a resposta gerada contém termos-chave esperados (ex.: número do artigo, valor numérico). É rápida e objetiva, mas não captura qualidade semântica.
2. **Análise qualitativa:** comparação manual entre a resposta esperada e a gerada, observando precisão, completude, citação de artigos e clareza.

## Dataset de Avaliação

O arquivo `tests/questions_benchmark.json` contém **10 perguntas** cobrindo os principais temas trabalhistas:
férias, aviso prévio, jornada de trabalho, hora extra, FGTS, licença maternidade,
intervalo intrajornada, rescisão, salário mínimo e trabalho noturno.

---

## 1. Configuração do Ambiente

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

# Adiciona a raiz do projeto ao sys.path para importar src.*
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.retrieval.chain import get_answer

print(f"Raiz do projeto: {project_root}")
print("Módulos carregados com sucesso.")

## 2. Carregamento do Benchmark

In [ ]:
benchmark_path = project_root / "tests" / "questions_benchmark.json"

with open(benchmark_path, encoding="utf-8") as f:
    benchmark = json.load(f)

print(f"{len(benchmark)} perguntas carregadas.")
print(f"Temas: {', '.join(item['topic'] for item in benchmark)}")

## 3. Execução dos Testes

Cada pergunta é avaliada de forma isolada (`chat_history=[]`), simulando uma consulta nova ao sistema.

In [ ]:
results = []

for item in benchmark:
    # Avaliação isolada: sem histórico de conversa
    generated = get_answer(item["question"], chat_history=[])

    keyword_hit = any(
        kw.lower() in generated.lower() for kw in item["expected_keywords"]
    )

    results.append({
        "id": item["id"],
        "topic": item["topic"],
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "expected_keywords": item["expected_keywords"],
        "generated_answer": generated,
        "keyword_hit": keyword_hit,
    })

    status = "✅" if keyword_hit else "❌"
    print(f"{status} [{item['topic']:25s}] {item['question'][:60]}...")

## 4. Métricas de Avaliação Automática

In [ ]:
total = len(results)
hits = sum(r["keyword_hit"] for r in results)
accuracy = hits / total

print(f"Total de perguntas : {total}")
print(f"Acertos (keyword)  : {hits}")
print(f"Erros              : {total - hits}")
print(f"Keyword Accuracy   : {accuracy:.0%}")

In [ ]:
df = pd.DataFrame(results)[[
    "id", "topic", "question", "keyword_hit", "expected_keywords"
]]
df["keyword_hit"] = df["keyword_hit"].map({True: "✅", False: "❌"})
df["expected_keywords"] = df["expected_keywords"].apply(", ".join)
df.set_index("id", inplace=True)

pd.set_option("display.max_colwidth", 80)
df

## 5. Comparação Resposta Esperada × Resposta Gerada

In [ ]:
for r in results:
    status = "✅" if r["keyword_hit"] else "❌"
    print(f"{'='*70}")
    print(f"{status} Pergunta #{r['id']} — {r['topic'].upper()}")
    print(f"\nP: {r['question']}")
    print(f"\n[ESPERADO]\n{r['expected_answer']}")
    print(f"\n[GERADO]\n{r['generated_answer']}")
    print()

## 6. Análise Qualitativa

Preencha esta seção após executar as células acima, analisando os resultados obtidos.

### 6.1 Pontos Fortes Observados

- *(ex.: o modelo citou corretamente o número do artigo na maioria das perguntas)*
- *(ex.: linguagem clara e acessível, sem jargão excessivo)*
- *(ex.: respostas sobre jornada e hora extra foram precisas e completas)*

### 6.2 Pontos de Melhoria

- *(ex.: perguntas sobre FGTS geraram respostas genéricas sem o percentual correto)*
- *(ex.: o modelo às vezes cita informações que não estão nos trechos recuperados)*
- *(ex.: respostas muito longas para perguntas diretas)*

### 6.3 Casos de Falha

| # | Pergunta | Motivo do Erro |
|---|----------|----------------|
| - | -        | -              |

### 6.4 Hipóteses para os Erros

- **Chunking insuficiente:** o trecho relevante não foi recuperado pelo retriever (k muito baixo ou embedding fraco)
- **Prompt insuficiente:** o modelo não foi guiado a citar o artigo explicitamente
- **Ambiguidade da pergunta:** a pergunta pode ser interpretada de múltiplas formas

### 6.5 Próximos Passos

- [ ] Aumentar `k` no retriever para perguntas com baixa keyword accuracy
- [ ] Avaliar embeddings alternativos (ex.: `text-embedding-004` vs `text-embedding-3-small`)
- [ ] Implementar avaliação semântica com `sentence-transformers` (BLEU, ROUGE ou cosine similarity)
- [ ] Adicionar mais perguntas ao benchmark, incluindo casos-limite e perguntas ambíguas

---

### 6.6 Resultado Final

| Métrica               | Valor |
|-----------------------|-------|
| Keyword Accuracy      | ?/10  |
| Perguntas com artigo  | ?/10  |
| Perguntas completas   | ?/10  |
| Nota qualitativa geral| ? / 5 |